# Can we predict late delivery?
Follows protocol.yaml. `claim_paid_eur` is excluded: a claim settled for the consignment is recorded after the outcome (leakage). `consignment` (id) and `dispatched_on` (split only) are excluded per protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
df=pd.read_csv('data.csv'); df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
print(len(df),len(train_idx),len(hold_idx))

In [ ]:
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']; FEATURES=NUM+CAT
# training rows sorted by time for walk-forward CV
train=df.iloc[train_idx].sort_values('dispatched_on',kind='stable')
X,y=train[FEATURES],train['late_delivery']; train_rows=train.index.to_numpy()
def pre(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
# all candidates named before any is scored
candidates={
 'baseline_prior':Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'hgb':Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'rf':Pipeline([('p',pre(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
}
print(list(candidates))

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
folds=list(tscv.split(X))
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(X.iloc[tr],y.iloc[tr])
        s.append(float(roc_auc_score(y.iloc[va],mm.predict_proba(X.iloc[va])[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); print(cv.round(4))
chosen=cv['mean'].idxmax(); print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(X,y)
H=df.iloc[hold_idx]
auc=float(roc_auc_score(H['late_delivery'],final.predict_proba(H[FEATURES])[:,1]))
print('holdout roc_auc (scored once):',round(auc,4))

In [ ]:
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[[int(train_rows[i]) for i in tr],[int(train_rows[i]) for i in va]] for tr,va in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':[int(i) for i in hold_idx],
 'holdout_scores':{chosen:auc},
}
json.dump(evidence,open('evidence.json','w'))